# 05 - Training

Load model YOLOv8s (như `04_model.ipynb`), cấu hình training theo `configs/training.yaml`, và fine-tune
trên `data/processed/data.yaml` (6.061 ảnh, single-class `Human`).

**Quy trình 2 bước**:
1. **Smoke test** (1-2 epoch) - xác nhận pipeline chạy được, không OOM, logging/checkpoint đúng chỗ, trước khi cam kết chạy full training (có thể mất nhiều giờ trên laptop GPU 6GB VRAM).
2. **Full training** (100 epoch theo config) - cell riêng, chỉ chạy khi đã xác nhận smoke test ổn và ước tính thời gian chấp nhận được.

## 0. Setup

Đọc `configs/training.yaml` + `configs/model.yaml`, build model như notebook 04.

In [ ]:
import os
import sys
import shutil
import datetime

import yaml
import torch

sys.path.insert(0, os.path.abspath(".."))
from src.models.model_factory import build_model

with open("../configs/training.yaml", encoding="utf-8") as f:
    train_cfg = yaml.safe_load(f)
with open("../configs/model.yaml", encoding="utf-8") as f:
    model_cfg = yaml.safe_load(f)

data_yaml_path = os.path.abspath("../data/processed/data.yaml")
if not os.path.isfile(data_yaml_path):
    raise RuntimeError("Chưa có data/processed/data.yaml. Chạy notebook 03_preprocessing.ipynb trước.")

weights_dir = os.path.abspath("../outputs/checkpoints")
logs_dir = os.path.abspath("../outputs/logs")  # chua ca weights/results/tensorboard cua tung run

device = 0 if torch.cuda.is_available() else "cpu"
print("Device:", device, "-", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")
print("data.yaml:", data_yaml_path)
print("Training config:", train_cfg)

In [ ]:
model = build_model(model_cfg, weights_dir=weights_dir)
print("Model:", model_cfg["name"], model_cfg["variant"], "| pretrained:", model_cfg["pretrained"])

## 1. Ánh xạ `configs/training.yaml` sang tham số `model.train()`

**Lưu ý về early stopping**: `configs/training.yaml` khai báo `monitor: val_map50`, nhưng ultralytics dùng
một chỉ số nội bộ gọi là *fitness* (kết hợp có trọng số giữa mAP50 và mAP50-95, không phải mAP50 đơn thuần)
để quyết định early stop qua tham số `patience` - không có tuỳ chọn giám sát đúng 1 metric string tuỳ ý
trong API chuẩn. Vẫn dùng `patience` từ config, nhưng ghi nhận sai khác này để không hiểu lầm là early stop
chính xác theo mAP50 riêng lẻ.

In [ ]:
def build_train_kwargs(epochs, project, name):
    return dict(
        data=data_yaml_path,
        epochs=epochs,
        imgsz=train_cfg["training"]["image_size"][0],
        batch=train_cfg["training"]["batch_size"],
        workers=train_cfg["training"]["num_workers"],
        amp=train_cfg["training"]["amp"],
        optimizer=train_cfg["optimizer"]["name"].upper(),
        lr0=train_cfg["optimizer"]["lr"],
        weight_decay=train_cfg["optimizer"]["weight_decay"],
        cos_lr=(train_cfg["scheduler"]["name"] == "cosine"),
        warmup_epochs=train_cfg["scheduler"]["warmup_epochs"],
        patience=train_cfg["early_stopping"]["patience"],
        seed=train_cfg["seed"],
        device=device,
        project=project,
        name=name,
        exist_ok=True,
        verbose=True,
    )

print(build_train_kwargs(epochs=1, project=logs_dir, name="_demo"))

## 2. Smoke test (2 epoch)

Chạy training thật nhưng chỉ 2 epoch trên toàn bộ tập train/val - mục tiêu là xác nhận:
- Không tràn VRAM (OOM) ở `batch=16`, `imgsz=1280` trên GPU 6GB
- Data loader đọc đúng ảnh/label từ `data/processed/`
- Checkpoint + log ghi đúng chỗ dưới `outputs/logs/`

Nếu OOM, tự động thử lại với batch nhỏ hơn (8, rồi 4) để tìm batch size khả thi trên máy này.

In [ ]:
SMOKE_EPOCHS = 2
smoke_name = "yolov8s_thermal_smoke"

batch_candidates = [train_cfg["training"]["batch_size"], 8, 4]
used_batch = None
smoke_start = datetime.datetime.now()

for batch in batch_candidates:
    kwargs = build_train_kwargs(epochs=SMOKE_EPOCHS, project=logs_dir, name=smoke_name)
    kwargs["batch"] = batch
    try:
        smoke_results = model.train(**kwargs)
        used_batch = batch
        break
    except torch.cuda.OutOfMemoryError:
        print(f"OOM voi batch={batch}, thu batch nho hon...")
        torch.cuda.empty_cache()
        continue

if used_batch is None:
    raise RuntimeError("OOM ngay ca voi batch=4 - can giam imgsz hoac dung model nho hon (yolov8n).")

smoke_elapsed = (datetime.datetime.now() - smoke_start).total_seconds()
print(f"\nSmoke test xong: batch={used_batch}, {SMOKE_EPOCHS} epoch trong {smoke_elapsed:.1f}s "
      f"(~{smoke_elapsed / SMOKE_EPOCHS:.1f}s/epoch)")
if used_batch != train_cfg["training"]["batch_size"]:
    print(f"CANH BAO: batch_size trong configs/training.yaml ({train_cfg['training']['batch_size']}) "
          f"gay OOM tren GPU nay - da tu dong giam xuong {used_batch}. Nen cap nhat lai config.")

## 3. Ước tính thời gian full training

In [ ]:
per_epoch_sec = smoke_elapsed / SMOKE_EPOCHS
full_epochs = train_cfg["training"]["epochs"]
estimated_total_sec = per_epoch_sec * full_epochs
estimated_hours = estimated_total_sec / 3600

print(f"Batch size se dung cho full training: {used_batch}")
print(f"Uoc tinh: {full_epochs} epoch x ~{per_epoch_sec:.1f}s/epoch = ~{estimated_hours:.1f} gio "
      f"(chua tinh early stopping co the dung som hon neu patience={train_cfg['early_stopping']['patience']} "
      f"epoch khong cai thien)."
)
print("=> Xem lai uoc tinh nay truoc khi chay cell '4. Full training' ben duoi.")

## 4. Full training (100 epoch theo config)

**Chỉ chạy cell dưới khi đã xem ước tính thời gian ở mục 3 và chấp nhận thời lượng đó.**
Dùng `batch` đã xác nhận không OOM ở bước smoke test. Kết quả (weights, log, plot) lưu tại
`outputs/logs/yolov8s_thermal/`; sau khi xong, `best.pt` được copy sang `outputs/checkpoints/best.pt`
để khớp với `configs/inference.yaml` (`checkpoint: outputs/checkpoints/best.pt`).

In [ ]:
RUN_FULL_TRAINING = False  # doi thanh True khi da san sang chay that (co the mat nhieu gio)

if not RUN_FULL_TRAINING:
    print("RUN_FULL_TRAINING=False -> bo qua. Doi thanh True o cell tren de chay full training that.")
else:
    full_name = "yolov8s_thermal"
    kwargs = build_train_kwargs(epochs=full_epochs, project=logs_dir, name=full_name)
    kwargs["batch"] = used_batch
    full_start = datetime.datetime.now()
    full_results = model.train(**kwargs)
    full_elapsed = (datetime.datetime.now() - full_start).total_seconds()
    print(f"Full training xong trong {full_elapsed / 3600:.2f} gio")

    best_src = os.path.join(logs_dir, full_name, "weights", "best.pt")
    checkpoint_dst = os.path.join(weights_dir, "best.pt")
    if os.path.isfile(best_src):
        shutil.copy2(best_src, checkpoint_dst)
        print("Da copy best checkpoint sang:", checkpoint_dst)
    else:
        print("CANH BAO: khong tim thay", best_src)

## 5. Tóm tắt

In [ ]:
print("=" * 60)
print("TÓM TẮT TRAINING")
print("=" * 60)
print(f"Smoke test: {SMOKE_EPOCHS} epoch, batch={used_batch}, {smoke_elapsed:.1f}s "
      f"(~{per_epoch_sec:.1f}s/epoch) - PASS, khong OOM")
print(f"Uoc tinh full training ({full_epochs} epoch): ~{estimated_hours:.1f} gio")
print(f"RUN_FULL_TRAINING = {RUN_FULL_TRAINING}")
print(f"Log/checkpoint tai: {logs_dir}")
print(f"Checkpoint cuoi cung (khi full training xong): {os.path.join(weights_dir, 'best.pt')}")